In [3]:
from collections import defaultdict
import glob
import os
import re
import xarray as xr

In [4]:
# --- Configuration Paths ---
input_dir = "/share/home/c1755103/dataset/ERA"
output_dir = "/share/home/c1755103/dataset/ERA/"
os.makedirs(output_dir, exist_ok=True)

# 1. Gather all netCDF files
all_files = glob.glob(os.path.join(input_dir, "HAD_pet_*.nc"))
print(f"Found {len(all_files)} monthly files in {input_dir}.")
# 2. Match strict monthly pattern: HAD_pet_YYYY_MM.nc (capturing YYYY)
monthly_pattern = re.compile(r"HAD_pet_(\d{4})_\d{2}\.nc$")

# Group files by their year using a dictionary
yearly_groups = defaultdict(list)
for file_path in all_files:
    filename = os.path.basename(file_path)
    match = monthly_pattern.search(filename)
    if match:
        year = match.group(1)  # Extracts the YYYY part
        yearly_groups[year].append(file_path)

# 3. Process and combine each year individually
if not yearly_groups:
    print("No matching monthly files found. Check your directory paths.")
else:
    print(f"Found {len(yearly_groups)} distinct years to process.")

    for year, files in sorted(yearly_groups.items()):
        # Sort files chronologically (Jan to Dec)
        sorted_files = sorted(files)
        print(f"\nProcessing year {year} ({len(sorted_files)} months found)...")

        output_file = os.path.join(output_dir, f"HAD_pet_H_{year}.nc")

        try:
            # Open and combine all 12 months for this year lazily
            ds_yearly = xr.open_mfdataset(sorted_files, concat_dim="time",
                                          combine="nested", chunks={"time": 1})
            
            # rename "valid_time" to "time" for clarity
            #ds_yearly = ds_yearly.rename({"valid_time": "time"})

            # Save out to a clean yearly file
            ds_yearly.to_netcdf(output_file)
            print(f"--> Successfully saved: {output_file}")

            # Close file handle to free system RAM
            ds_yearly.close()

        except Exception as e:
            print(f"--> Error combining year {year}: {e}")



Found 33 monthly files in /share/home/c1755103/dataset/ERA.
Found 1 distinct years to process.

Processing year 2026 (6 months found)...
--> Successfully saved: /share/home/c1755103/dataset/ERA/HAD_pet_H_2026.nc


In [10]:
path = "/share/home/c1755103/dataset/ERA/HAD_pet_H_2026.nc"
# open the dataset to check its contents
ds = xr.open_dataset(path)
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 3900, latitude: 225, longitude: 251)
Coordinates:
    number     int64 8B ...
  * time       (time) datetime64[ns] 31kB 2026-01-01 ... 2026-06-12T11:00:00
  * latitude   (latitude) float64 2kB 15.5 15.4 15.3 15.2 ... -6.7 -6.8 -6.9
  * longitude  (longitude) float64 2kB 28.1 28.2 28.3 28.4 ... 52.9 53.0 53.1
    expver     (time) <U4 62kB ...
Data variables:
    pet        (time, latitude, longitude) float64 2GB ...


In [3]:
path = "/share/home/c1755103/dataset/ERA/HAD_pet_2026_06.nc"
# check the contents of the dataset
ds = xr.open_dataset(path)
print(ds)

<xarray.Dataset> Size: 125MB
Dimensions:    (time: 276, latitude: 225, longitude: 251)
Coordinates:
    number     int64 8B ...
  * time       (time) datetime64[ns] 2kB 2026-06-01 ... 2026-06-12T11:00:00
  * latitude   (latitude) float64 2kB 15.5 15.4 15.3 15.2 ... -6.7 -6.8 -6.9
  * longitude  (longitude) float64 2kB 28.1 28.2 28.3 28.4 ... 52.9 53.0 53.1
    expver     (time) <U4 4kB ...
Data variables:
    pet        (time, latitude, longitude) float64 125MB ...
